# 📊 NBA Exploratory Data Analysis

**Purpose**: Deep dive analysis of team performance, player statistics, and betting market patterns  
**Data Sources**: SQLite database populated by ingestion pipeline  
**Visualizations**: Interactive plots, correlation heatmaps, performance dashboards  
**Insights**: Identify betting edges, team trends, market inefficiencies  

---

## 🔧 Setup & Data Loading

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from datetime import datetime, timedelta
import scipy.stats as stats
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Configure plotting
plt.style.use('dark_background')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Set up larger plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

print("🏀 NBA EDA Environment Ready!")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Load data from SQLite database
def load_nba_data(db_path='../../data/nba_data.db'):
    """
    Load all NBA data from SQLite database
    Returns dictionary of DataFrames
    """
    try:
        conn = sqlite3.connect(db_path)
        
        # Load all tables
        tables = {
            'schedule': pd.read_sql_query("SELECT * FROM nba_schedule", conn),
            'odds': pd.read_sql_query("SELECT * FROM nba_odds", conn),
            'team_stats': pd.read_sql_query("SELECT * FROM nba_team_stats", conn),
            'injuries': pd.read_sql_query("SELECT * FROM nba_injuries", conn)
        }
        
        conn.close()
        
        print(f"✅ Loaded data:")
        for table, df in tables.items():
            print(f"  {table}: {len(df):,} records")
        
        return tables
        
    except Exception as e:
        print(f"❌ Error loading data: {e}")
        # Return empty DataFrames if database doesn't exist
        return {
            'schedule': pd.DataFrame(),
            'odds': pd.DataFrame(), 
            'team_stats': pd.DataFrame(),
            'injuries': pd.DataFrame()
        }

# Load all NBA data
nba_data = load_nba_data()
schedule_df = nba_data['schedule']
odds_df = nba_data['odds']
team_stats_df = nba_data['team_stats']
injuries_df = nba_data['injuries']

print(f"\n📊 Data Overview:")
print(f"🗓️ Schedule: {len(schedule_df)} games")
print(f"💰 Odds: {len(odds_df)} betting lines")
print(f"🏀 Team Stats: {len(team_stats_df)} team records")
print(f"🏥 Injuries: {len(injuries_df)} reports")

## 🏀 Team Performance Analysis

In [ ]:
# Team performance correlation analysis
if not team_stats_df.empty:
    # Select key performance metrics
    performance_metrics = ['PTS', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF']
    
    # Filter to numeric columns that exist
    available_metrics = [col for col in performance_metrics if col in team_stats_df.columns]
    
    if available_metrics:
        # Create correlation heatmap
        plt.figure(figsize=(14, 10))
        correlation_matrix = team_stats_df[available_metrics].corr()
        
        sns.heatmap(correlation_matrix, 
                   annot=True, 
                   cmap='RdYlBu_r', 
                   center=0, 
                   square=True,
                   fmt='.2f',
                   cbar_kws={'shrink': 0.8})
        
        plt.title('🏀 NBA Team Performance Correlation Matrix', fontsize=16, pad=20)
        plt.tight_layout()
        plt.show()
        
        print(f"🔍 Strong Correlations (|r| > 0.7):")
        strong_corrs = []
        for i, col1 in enumerate(available_metrics):
            for j, col2 in enumerate(available_metrics[i+1:], i+1):
                corr_val = correlation_matrix.loc[col1, col2]
                if abs(corr_val) > 0.7:
                    strong_corrs.append((col1, col2, corr_val))
        
        for col1, col2, corr in sorted(strong_corrs, key=lambda x: abs(x[2]), reverse=True):
            print(f"  {col1} ↔ {col2}: {corr:.3f}")
    else:
        print("⚠️ No performance metrics available for correlation analysis")
else:
    print("⚠️ No team stats data available")

In [ ]:
# Team efficiency scatter plots
if not team_stats_df.empty and 'PTS' in team_stats_df.columns:
    
    # Create interactive scatter plot
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Offensive Efficiency', 'Defensive Rating', 'Pace vs Scoring', 'Shooting Efficiency'),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # Plot 1: Points vs Field Goal %
    if 'FG_PCT' in team_stats_df.columns:
        fig.add_trace(
            go.Scatter(
                x=team_stats_df['FG_PCT'],
                y=team_stats_df['PTS'],
                mode='markers+text',
                text=team_stats_df.get('TEAM_NAME', team_stats_df.index),
                textposition='top center',
                marker=dict(size=10, color='lightblue'),
                name='Teams'
            ),
            row=1, col=1
        )
    
    # Plot 2: Rebounds vs Assists
    if all(col in team_stats_df.columns for col in ['REB', 'AST']):
        fig.add_trace(
            go.Scatter(
                x=team_stats_df['REB'],
                y=team_stats_df['AST'],
                mode='markers',
                marker=dict(size=10, color='lightgreen'),
                name='REB vs AST'
            ),
            row=1, col=2
        )
    
    # Plot 3: Turnovers vs Steals
    if all(col in team_stats_df.columns for col in ['TOV', 'STL']):
        fig.add_trace(
            go.Scatter(
                x=team_stats_df['TOV'],
                y=team_stats_df['STL'],
                mode='markers',
                marker=dict(size=10, color='orange'),
                name='TOV vs STL'
            ),
            row=2, col=1
        )
    
    # Plot 4: 3P% vs FT%
    if all(col in team_stats_df.columns for col in ['FG3_PCT', 'FT_PCT']):
        fig.add_trace(
            go.Scatter(
                x=team_stats_df['FG3_PCT'],
                y=team_stats_df['FT_PCT'],
                mode='markers',
                marker=dict(size=10, color='red'),
                name='3P% vs FT%'
            ),
            row=2, col=2
        )
    
    fig.update_layout(
        title_text='🏀 NBA Team Performance Analysis Dashboard',
        showlegend=False,
        height=800
    )
    
    fig.show()
    
else:
    print("⚠️ Insufficient team stats data for efficiency analysis")

## 💰 Betting Market Analysis

In [ ]:
# Odds distribution and market analysis
if not odds_df.empty:
    
    # Analyze odds by market type
    print("💰 Betting Market Overview:")
    market_summary = odds_df.groupby('market_type').agg({
        'bookmaker': 'nunique',
        'price': ['count', 'mean', 'std'],
        'game_id': 'nunique'
    }).round(2)
    
    market_summary.columns = ['Bookmakers', 'Total_Lines', 'Avg_Price', 'Price_Std', 'Games']
    display(market_summary)
    
    # Moneyline odds distribution
    if 'h2h' in odds_df['market_type'].values:
        moneyline_df = odds_df[odds_df['market_type'] == 'h2h'].copy()
        
        plt.figure(figsize=(15, 10))
        
        # Subplot 1: Odds distribution by bookmaker
        plt.subplot(2, 2, 1)
        moneyline_df.boxplot(column='price', by='bookmaker', ax=plt.gca())
        plt.title('💰 Moneyline Odds by Bookmaker')
        plt.xlabel('Bookmaker')
        plt.ylabel('American Odds')
        plt.xticks(rotation=45)
        
        # Subplot 2: Odds histogram
        plt.subplot(2, 2, 2)
        plt.hist(moneyline_df['price'], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
        plt.title('📊 Moneyline Odds Distribution')
        plt.xlabel('American Odds')
        plt.ylabel('Frequency')
        
        # Subplot 3: Favorite vs Underdog split
        plt.subplot(2, 2, 3)
        favorites = moneyline_df[moneyline_df['price'] < 0]['price']
        underdogs = moneyline_df[moneyline_df['price'] > 0]['price']
        
        plt.hist([favorites, underdogs], bins=20, alpha=0.7, 
                label=['Favorites (<0)', 'Underdogs (>0)'], color=['red', 'green'])
        plt.title('⚖️ Favorites vs Underdogs')
        plt.xlabel('American Odds')
        plt.ylabel('Count')
        plt.legend()
        
        # Subplot 4: Bookmaker price variance
        plt.subplot(2, 2, 4)
        price_variance = moneyline_df.groupby(['game_id', 'team'])['price'].agg(['min', 'max', 'std']).reset_index()
        price_variance['range'] = price_variance['max'] - price_variance['min']
        
        plt.scatter(price_variance['std'], price_variance['range'], alpha=0.6, color='purple')
        plt.title('📈 Price Variance Across Books')
        plt.xlabel('Standard Deviation')
        plt.ylabel('Price Range (Max - Min)')
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n🎯 Market Insights:")
        print(f"  📊 Total betting lines: {len(moneyline_df):,}")
        print(f"  🏪 Active bookmakers: {moneyline_df['bookmaker'].nunique()}")
        print(f"  ⚖️ Favorites: {len(favorites):,} | Underdogs: {len(underdogs):,}")
        print(f"  💰 Avg favorite odds: {favorites.mean():.0f}")
        print(f"  🎯 Avg underdog odds: {underdogs.mean():.0f}")
        
else:
    print("⚠️ No odds data available for market analysis")

In [ ]:
# Identify arbitrage opportunities
def find_arbitrage_opportunities(odds_df):
    """
    Find potential arbitrage opportunities in moneyline betting
    """
    if odds_df.empty or 'h2h' not in odds_df['market_type'].values:
        return pd.DataFrame()
    
    moneyline = odds_df[odds_df['market_type'] == 'h2h'].copy()
    
    # Convert American odds to decimal
    def american_to_decimal(odds):
        if odds > 0:
            return (odds / 100) + 1
        else:
            return (100 / abs(odds)) + 1
    
    moneyline['decimal_odds'] = moneyline['price'].apply(american_to_decimal)
    moneyline['implied_prob'] = 1 / moneyline['decimal_odds']
    
    # Find best odds for each team
    arbitrage_opps = []
    
    for game_id in moneyline['game_id'].unique():
        game_odds = moneyline[moneyline['game_id'] == game_id]
        
        for team in game_odds['team'].unique():
            team_odds = game_odds[game_odds['team'] == team]
            best_odds = team_odds.loc[team_odds['decimal_odds'].idxmax()]
            
            # Check for arbitrage against other team
            opponent_odds = game_odds[game_odds['team'] != team]
            if not opponent_odds.empty:
                best_opponent = opponent_odds.loc[opponent_odds['decimal_odds'].idxmax()]
                
                # Calculate arbitrage percentage
                total_implied = best_odds['implied_prob'] + best_opponent['implied_prob']
                
                if total_implied < 1.0:  # Arbitrage opportunity
                    arbitrage_opps.append({
                        'game_id': game_id,
                        'team_1': best_odds['team'],
                        'team_1_odds': best_odds['price'],
                        'team_1_book': best_odds['bookmaker'],
                        'team_2': best_opponent['team'],
                        'team_2_odds': best_opponent['price'],
                        'team_2_book': best_opponent['bookmaker'],
                        'arbitrage_pct': (1 - total_implied) * 100,
                        'profit_per_100': ((1 - total_implied) / total_implied) * 100
                    })
    
    return pd.DataFrame(arbitrage_opps)

# Find arbitrage opportunities
arbitrage_df = find_arbitrage_opportunities(odds_df)

if not arbitrage_df.empty:
    print("🎯 Potential Arbitrage Opportunities:")
    arbitrage_df_sorted = arbitrage_df.sort_values('arbitrage_pct', ascending=False)
    display(arbitrage_df_sorted.head())
    
    if len(arbitrage_df_sorted) > 0:
        print(f"\n💰 Best Arbitrage:")
        best = arbitrage_df_sorted.iloc[0]
        print(f"  🏀 {best['team_1']} ({best['team_1_odds']}) vs {best['team_2']} ({best['team_2_odds']})")
        print(f"  📚 Books: {best['team_1_book']} vs {best['team_2_book']}")
        print(f"  💸 Profit: {best['arbitrage_pct']:.2f}% ({best['profit_per_100']:.2f} per $100)")
else:
    print("ℹ️ No arbitrage opportunities found in current data")

## 🔍 Team Clustering & Performance Profiles

In [ ]:
# Cluster teams based on performance metrics
if not team_stats_df.empty:
    # Select clustering features
    cluster_features = ['PTS', 'FG_PCT', 'FG3_PCT', 'REB', 'AST', 'STL', 'BLK', 'TOV']
    available_features = [col for col in cluster_features if col in team_stats_df.columns]
    
    if len(available_features) >= 4:
        # Prepare data for clustering
        cluster_data = team_stats_df[available_features].fillna(0)
        
        # Standardize features
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(cluster_data)
        
        # Perform K-means clustering
        n_clusters = min(5, len(team_stats_df) // 2)  # Max 5 clusters, at least 2 teams per cluster
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        team_stats_df['cluster'] = kmeans.fit_predict(scaled_data)
        
        # Analyze clusters
        print(f"🔍 Team Performance Clusters (K={n_clusters}):")
        
        cluster_summary = team_stats_df.groupby('cluster').agg({
            **{feat: 'mean' for feat in available_features},
            'TEAM_NAME': 'count'
        }).round(2)
        
        cluster_summary.columns = available_features + ['Team_Count']
        display(cluster_summary)
        
        # Visualize clusters
        if len(available_features) >= 2:
            fig = px.scatter(
                team_stats_df,
                x=available_features[0],
                y=available_features[1],
                color='cluster',
                text='TEAM_NAME' if 'TEAM_NAME' in team_stats_df.columns else team_stats_df.index,
                title=f"🏀 Team Clusters: {available_features[0]} vs {available_features[1]}",
                hover_data=available_features[:4]  # Show top 4 features on hover
            )
            
            fig.update_traces(textposition='top center')
            fig.update_layout(height=600)
            fig.show()
        
        # Define cluster archetypes
        cluster_archetypes = {
            0: "Balanced Teams",
            1: "Offensive Powerhouses", 
            2: "Defensive Specialists",
            3: "High-Tempo Teams",
            4: "Developing Teams"
        }
        
        print("\n🏆 Team Archetypes:")
        for cluster_id in sorted(team_stats_df['cluster'].unique()):
            cluster_teams = team_stats_df[team_stats_df['cluster'] == cluster_id]
            archetype = cluster_archetypes.get(cluster_id, f"Cluster {cluster_id}")
            teams = cluster_teams.get('TEAM_NAME', cluster_teams.index).tolist()
            
            print(f"  🎯 {archetype} ({len(teams)} teams):")
            print(f"     {', '.join(teams)}")
    else:
        print("⚠️ Insufficient features for clustering analysis")
else:
    print("⚠️ No team stats data available for clustering")

## 🏥 Injury Impact Analysis

In [ ]:
# Analyze injury reports and impact on team performance
if not injuries_df.empty:
    
    print("🏥 Current Injury Landscape:")
    
    # Injury status distribution
    injury_status = injuries_df['status'].value_counts()
    print(f"\n📊 Injury Status Breakdown:")
    for status, count in injury_status.items():
        print(f"  {status}: {count} players")
    
    # Teams with most injuries
    team_injuries = injuries_df['team'].value_counts()
    print(f"\n🚑 Teams with Most Injuries:")
    for team, count in team_injuries.head().items():
        print(f"  {team}: {count} players")
    
    # Visualize injury impact
    plt.figure(figsize=(15, 8))
    
    # Subplot 1: Status distribution
    plt.subplot(2, 2, 1)
    injury_status.plot(kind='bar', color=['red', 'orange', 'yellow', 'green'][:len(injury_status)])
    plt.title('🏥 Injury Status Distribution')
    plt.xlabel('Status')
    plt.ylabel('Number of Players')
    plt.xticks(rotation=45)
    
    # Subplot 2: Team impact
    plt.subplot(2, 2, 2)
    team_injuries.head(10).plot(kind='barh', color='lightcoral')
    plt.title('🚑 Teams by Injury Count')
    plt.xlabel('Number of Injured Players')
    
    # Subplot 3: Injury types
    plt.subplot(2, 2, 3)
    if 'injury' in injuries_df.columns:
        injury_types = injuries_df['injury'].value_counts().head(8)
        injury_types.plot(kind='pie', autopct='%1.1f%%', startangle=90)
        plt.title('🩹 Common Injury Types')
        plt.ylabel('')
    
    plt.tight_layout()
    plt.show()
    
    # High-impact injury alerts
    key_players = injuries_df[injuries_df['status'].isin(['Out', 'Doubtful'])]
    if not key_players.empty:
        print(f"\n⚠️ High-Impact Injury Alerts:")
        for _, player in key_players.iterrows():
            print(f"  🚨 {player['player']} ({player['team']}): {player['status']} - {player.get('injury', 'Unknown')}")
    
else:
    print("ℹ️ No injury data available for analysis")

## 📈 Market Efficiency Analysis

In [ ]:
# Analyze betting market efficiency and identify potential edges
def calculate_market_efficiency(odds_df, team_stats_df):
    """
    Calculate market efficiency by comparing implied probabilities to team performance
    """
    if odds_df.empty or team_stats_df.empty:
        return pd.DataFrame()
    
    # Focus on moneyline odds
    if 'h2h' not in odds_df['market_type'].values:
        return pd.DataFrame()
    
    moneyline = odds_df[odds_df['market_type'] == 'h2h'].copy()
    
    # Convert odds to implied probabilities
    def american_to_probability(odds):
        if odds > 0:
            return 100 / (odds + 100)
        else:
            return abs(odds) / (abs(odds) + 100)
    
    moneyline['implied_prob'] = moneyline['price'].apply(american_to_probability)
    
    # Calculate team strength metrics
    if 'PTS' in team_stats_df.columns:
        # Simple team strength based on points scored
        max_pts = team_stats_df['PTS'].max()
        min_pts = team_stats_df['PTS'].min()
        
        team_strength = {}
        for _, team in team_stats_df.iterrows():
            team_name = team.get('TEAM_NAME', str(team.name))
            if pd.notna(team['PTS']):
                strength = (team['PTS'] - min_pts) / (max_pts - min_pts)
                team_strength[team_name] = strength
        
        # Compare market vs performance
        efficiency_data = []
        
        for game_id in moneyline['game_id'].unique():
            game_odds = moneyline[moneyline['game_id'] == game_id]
            
            # Get average implied probability for each team
            for team in game_odds['team'].unique():
                team_odds = game_odds[game_odds['team'] == team]
                avg_implied_prob = team_odds['implied_prob'].mean()
                
                # Get team strength
                strength = team_strength.get(team, 0.5)  # Default to neutral
                
                efficiency_data.append({
                    'game_id': game_id,
                    'team': team,
                    'implied_prob': avg_implied_prob,
                    'team_strength': strength,
                    'market_bias': avg_implied_prob - strength,
                    'potential_edge': abs(avg_implied_prob - strength)
                })
        
        return pd.DataFrame(efficiency_data)
    
    return pd.DataFrame()

# Calculate market efficiency
efficiency_df = calculate_market_efficiency(odds_df, team_stats_df)

if not efficiency_df.empty:
    print("📈 Market Efficiency Analysis:")
    
    # Find biggest discrepancies
    biggest_edges = efficiency_df.nlargest(5, 'potential_edge')
    
    print(f"\n🎯 Potential Betting Edges:")
    for _, edge in biggest_edges.iterrows():
        direction = "OVERVALUED" if edge['market_bias'] > 0 else "UNDERVALUED"
        print(f"  🏀 {edge['team']}: {direction} by {edge['potential_edge']:.3f}")
        print(f"     Market: {edge['implied_prob']:.3f} | Strength: {edge['team_strength']:.3f}")
    
    # Visualize market efficiency
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 2, 1)
    plt.scatter(efficiency_df['team_strength'], efficiency_df['implied_prob'], 
               alpha=0.6, color='blue', s=60)
    plt.plot([0, 1], [0, 1], 'r--', label='Perfect Efficiency Line')
    plt.xlabel('Team Strength (Performance Based)')
    plt.ylabel('Market Implied Probability')
    plt.title('📊 Market vs Performance')
    plt.legend()
    
    plt.subplot(2, 2, 2)
    plt.hist(efficiency_df['market_bias'], bins=20, alpha=0.7, color='green', edgecolor='black')
    plt.axvline(x=0, color='red', linestyle='--', label='No Bias')
    plt.xlabel('Market Bias (Implied - Strength)')
    plt.ylabel('Frequency')
    plt.title('📈 Market Bias Distribution')
    plt.legend()
    
    plt.subplot(2, 2, 3)
    efficiency_df['potential_edge'].hist(bins=15, alpha=0.7, color='orange', edgecolor='black')
    plt.xlabel('Potential Edge Size')
    plt.ylabel('Frequency')
    plt.title('🎯 Edge Opportunities')
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics
    print(f"\n📊 Market Efficiency Summary:")
    print(f"  📈 Average edge size: {efficiency_df['potential_edge'].mean():.3f}")
    print(f"  🎯 Max edge opportunity: {efficiency_df['potential_edge'].max():.3f}")
    print(f"  ⚖️ Market bias (mean): {efficiency_df['market_bias'].mean():.3f}")
    print(f"  🎲 Opportunities > 0.1 edge: {len(efficiency_df[efficiency_df['potential_edge'] > 0.1])}")

else:
    print("ℹ️ Unable to calculate market efficiency - insufficient data")

## 📋 EDA Summary & Key Insights

In [ ]:
# Generate comprehensive EDA summary
print("🏀 NBA EDA Summary Report")
print("=" * 50)

print(f"\n📊 Data Coverage:")
print(f"  📅 Games analyzed: {len(schedule_df):,}")
print(f"  💰 Betting lines: {len(odds_df):,}")
print(f"  🏀 Teams tracked: {len(team_stats_df)}")
print(f"  🏥 Injury reports: {len(injuries_df)}")

if not odds_df.empty:
    print(f"\n💰 Betting Market:")
    print(f"  📚 Active bookmakers: {odds_df['bookmaker'].nunique()}")
    print(f"  🎯 Market types: {list(odds_df['market_type'].unique())}")
    
    if len(arbitrage_df) > 0:
        print(f"  🎯 Arbitrage opportunities: {len(arbitrage_df)}")
        print(f"  💸 Best arbitrage: {arbitrage_df['arbitrage_pct'].max():.2f}%")

if not team_stats_df.empty:
    print(f"\n🏀 Team Performance:")
    if 'PTS' in team_stats_df.columns:
        print(f"  ⭐ Highest scoring: {team_stats_df.loc[team_stats_df['PTS'].idxmax(), 'TEAM_NAME'] if 'TEAM_NAME' in team_stats_df.columns else 'Unknown'} ({team_stats_df['PTS'].max():.1f} PPG)")
        print(f"  🛡️ Lowest scoring: {team_stats_df.loc[team_stats_df['PTS'].idxmin(), 'TEAM_NAME'] if 'TEAM_NAME' in team_stats_df.columns else 'Unknown'} ({team_stats_df['PTS'].min():.1f} PPG)")
    
    if 'cluster' in team_stats_df.columns:
        print(f"  📊 Performance clusters: {team_stats_df['cluster'].nunique()}")

if not efficiency_df.empty:
    print(f"\n📈 Market Efficiency:")
    print(f"  🎯 Average edge size: {efficiency_df['potential_edge'].mean():.3f}")
    print(f"  💎 Max edge opportunity: {efficiency_df['potential_edge'].max():.3f}")
    strong_edges = efficiency_df[efficiency_df['potential_edge'] > 0.1]
    print(f"  🎲 Strong edges (>0.1): {len(strong_edges)}")

if not injuries_df.empty:
    print(f"\n🏥 Injury Impact:")
    out_players = injuries_df[injuries_df['status'] == 'Out']
    print(f"  🚨 Players out: {len(out_players)}")
    questionable = injuries_df[injuries_df['status'] == 'Questionable']
    print(f"  ❓ Questionable: {len(questionable)}")

print(f"\n🎯 Key Actionable Insights:")

insights = []

# Market insights
if len(arbitrage_df) > 0:
    insights.append(f"  💰 {len(arbitrage_df)} arbitrage opportunities detected")

# Performance insights  
if not efficiency_df.empty:
    strong_edges = efficiency_df[efficiency_df['potential_edge'] > 0.05]
    if len(strong_edges) > 0:
        insights.append(f"  📈 {len(strong_edges)} teams showing market inefficiency")

# Injury insights
if not injuries_df.empty:
    high_impact = injuries_df[injuries_df['status'].isin(['Out', 'Doubtful'])]
    if len(high_impact) > 0:
        insights.append(f"  🏥 {len(high_impact)} high-impact injury situations to monitor")

if not insights:
    insights = ["  ℹ️ Continue monitoring for opportunities as data updates"]

for insight in insights:
    print(insight)

print(f"\n⏰ Analysis completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🔄 Next EDA refresh recommended: {(datetime.now() + timedelta(hours=6)).strftime('%Y-%m-%d %H:%M:%S')}")